# TXPipe - Cluster shear calibrator

Test notebook for the `CLClusterShearCalibrator` stage (`txpipe/extensions/cluster_counts/cluster_shear_calibrator.py`).

The upstream stages of `tangential_shear_selection.ipynb` (source selection, cluster shear catalogs, cluster binning) are run once and their outputs are reused. After that, only the calibrator stage is run, and it is reloaded every time so edits to the stage are picked up without restarting the kernel.

In [ ]:
import os
import sys
import importlib
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import h5py
import ceci

In [ ]:
my_txpipe_dir = "/sps/lsst/users/ebarroso/TXPipe"
sys.path.append(my_txpipe_dir)
sys.path.append(f"{my_txpipe_dir}/txpipe/extensions")
import txpipe
import txpipe.extensions as extensions

## Files

All the outputs of this notebook go to `outputs_cluster_calibration`, so the other notebooks are not affected. Set `RERUN_UPSTREAM = True` to regenerate the upstream outputs.

In [ ]:
inputs_dir = f"{my_txpipe_dir}/data/example/inputs"
out_dir = f"{my_txpipe_dir}/data/example/outputs_cluster_calibration"
os.makedirs(out_dir, exist_ok=True)

shear_catalog = f"{inputs_dir}/metadetect_shear_catalog.hdf5"
cluster_catalog = f"{inputs_dir}/cluster_catalog.hdf5"
photoz_pdfs = f"{inputs_dir}/photoz_pdfs.hdf5"
fiducial_cosmology = f"{my_txpipe_dir}/data/fiducial_cosmology.yml"
shear_tomography_catalog = f"{my_txpipe_dir}/data/example/outputs_metadetect/shear_tomography_catalog.hdf5"

cluster_shear_catalogs = f"{out_dir}/cluster_shear_catalogs.hdf5"
cluster_catalog_tomography = f"{out_dir}/cluster_catalog_tomography.hdf5"
cluster_shear_catalogs_calibrated = f"{out_dir}/cluster_shear_catalogs_calibrated.hdf5"

RERUN_UPSTREAM = False

---

## Upstream stages (run once)

Same stages and options as `tangential_shear_selection.ipynb`, with one configuration only: $\Delta\Sigma$ with zmode for the cut and the weights.

In [ ]:
if RERUN_UPSTREAM or not os.path.exists(shear_tomography_catalog):
    step1 = txpipe.TXSourceSelectorMetadetect.make_stage(
        shear_catalog=shear_catalog,
        calibration_table=f"{inputs_dir}/sample_cosmodc2_w10year_errors.dat",
        config=f"{my_txpipe_dir}/examples/metadetect/config.yml",
        shear_tomography_catalog=shear_tomography_catalog,
        true_z=True,
    )
    step1.run()
    step1.finalize()

In [ ]:
if RERUN_UPSTREAM or not os.path.exists(cluster_shear_catalogs):
    step2 = extensions.CLClusterShearCatalogs.make_stage(
        shear_catalog=shear_catalog,
        cluster_catalog=cluster_catalog,
        fiducial_cosmology=fiducial_cosmology,
        shear_tomography_catalog=shear_tomography_catalog,
        source_photoz_pdfs=photoz_pdfs,
        cluster_shear_catalogs=cluster_shear_catalogs,
        output_dir=out_dir,
        max_radius=10.0,
        use_radius=True,
        max_angle=40,
        delta_sigma=True,
        redshift_weight_criterion="zmode",
        redshift_cut_criterion="zmode",
        use_shape_noise=False,
    )
    step2.run()
    step2.finalize()

In [ ]:
if RERUN_UPSTREAM or not os.path.exists(cluster_catalog_tomography):
    step3 = extensions.CLClusterBinningRedshiftRichness.make_stage(
        cluster_catalog=cluster_catalog,
        zedge=[0.2, 0.4, 0.6, 0.8],
        richedge=[5, 10, 100],
        cluster_catalog_tomography=cluster_catalog_tomography,
    )
    step3.run()
    step3.finalize()

---

## Calibrator stage

Rerun from here after every change to `cluster_shear_calibrator.py`. `allow_reload` lets ceci register the reloaded class again under the same name.

In [ ]:
ceci.PipelineStage.allow_reload = True
from txpipe.extensions.cluster_counts import cluster_shear_calibrator
importlib.reload(cluster_shear_calibrator)
CLClusterShearCalibrator = cluster_shear_calibrator.CLClusterShearCalibrator
print(CLClusterShearCalibrator.config_options)

In [ ]:
calib = CLClusterShearCalibrator.make_stage(
    cluster_catalog_tomography=cluster_catalog_tomography,
    cluster_shear_catalogs=cluster_shear_catalogs,
    fiducial_cosmology=fiducial_cosmology,
    shear_tomography_catalog=shear_tomography_catalog,
    cluster_shear_catalogs_calibrated=cluster_shear_catalogs_calibrated,
    output_dir=out_dir,
    r_min=0.2,
    r_max=3.0,
    nbins=5,
    bin_units="mpc",
)
calib.run()

## Clusters and sources of one richness-redshift bin

In [ ]:
with h5py.File(cluster_catalog_tomography, "r") as f:
    bin_names = list(f["cluster_bin"].keys())
print(bin_names)

bin_name = bin_names[0]
clusters, sources = calib.load_cluster_bin_sources(bin_name)
print(clusters.meta)
clusters

In [ ]:
print(sources.meta)
sources[:10]

Number of sources per cluster and radial bin. Sources with `radial_bin = -1` are outside `[r_min, r_max]`.

In [ ]:
nbins = len(calib.distance_bins) - 1
in_range = sources["radial_bin"] >= 0
print("Sources outside the radial range:", np.sum(~in_range))
print("Sources per radial bin:", np.bincount(sources["radial_bin"][in_range], minlength=nbins))

counts = np.zeros((len(clusters), nbins), dtype=int)
for i, cid in enumerate(clusters["cluster_id"]):
    sel = in_range & (sources["cluster_id"] == cid)
    counts[i] = np.bincount(sources["radial_bin"][sel], minlength=nbins)
for cid, c in zip(clusters["cluster_id"], counts):
    print(cid, c)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

for b in range(nbins):
    sel = sources["radial_bin"] == b
    ax[0].hist(sources["distance"][sel], bins=50, range=(0, sources["distance"].max()), histtype="step", label=f"bin {b}")
ax[0].hist(sources["distance"][~in_range], bins=50, range=(0, sources["distance"].max()), color="gray", alpha=0.3, label="outside")
for edge in calib.distance_bins:
    ax[0].axvline(edge, color="k", lw=0.5)
ax[0].set_xlabel(f"Distance [{sources.meta['bin_units']}]")
ax[0].set_ylabel("Sources")
ax[0].legend()

# Weighted mean of all the pairs of the bin, a quick look only (not the CLMM stacking)
r_mid = np.sqrt(calib.distance_bins[1:] * calib.distance_bins[:-1])
w = sources["weight"]
tan = [np.sum((w * sources["tangential_comp"])[sources["radial_bin"] == b]) / np.sum(w[sources["radial_bin"] == b]) for b in range(nbins)]
cross = [np.sum((w * sources["cross_comp"])[sources["radial_bin"] == b]) / np.sum(w[sources["radial_bin"] == b]) for b in range(nbins)]
ax[1].plot(r_mid, tan, "o-", label="tangential")
ax[1].plot(r_mid, cross, "s--", label="cross")
ax[1].set_xscale("log")
ax[1].set_xlabel(f"R [{sources.meta['bin_units']}]")
ax[1].set_ylabel(sources.meta["profile_type"])
ax[1].legend()
plt.tight_layout()
plt.show()

---

## Next: calibration

`source_index` is the row of each source in the shear catalog. The metadetect catalog has the groups below.

In [ ]:
with h5py.File(shear_catalog, "r") as f:
    for group in f["shear"]:
        print(group, f["shear"][group]["g1"].shape)

---

## Output catalog

The calibration is not applied yet (`calibrated = False`), so g1, g2 are the ones of `cluster_shear_catalogs`. Clusters and sources saved for each stack:

In [ ]:
with h5py.File(cluster_shear_catalogs_calibrated, "r") as f:
    print(dict(f["index"].attrs))
    index = {k: f["index"][k][:] for k in f["index"]}
    cluster_bin = f["catalog/cluster_bin"].asstr()[:]
    pair_bin = np.repeat(cluster_bin, f["catalog/cluster_sample_count"][:])
for name in np.unique(cluster_bin):
    print(name, "clusters =", np.sum(cluster_bin == name), "sources =", np.sum(pair_bin == name))

The pairs of `bin_name` in the output are the sources in `[r_min, r_max]` of `load_cluster_bin_sources`, in the same order.

In [ ]:
sel = pair_bin == bin_name
in_range = sources["radial_bin"] >= 0
for k in ["source_index", "radial_bin", "g1", "g2"]:
    print(k, np.array_equal(np.asarray(sources[k][in_range]), index[k][sel]))